In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType
from pyspark.sql import Row
from itertools import chain

### BUILD GOLD PRODUCT DIMENSION (gld_dim_products)

In [0]:
def build_dim_products():
    print("Building Gold Product Dimension...")
    
    df_products = spark.table(f"ecommerce.silver.slv_products")
    df_brands = spark.table(f"ecommerce.silver.slv_brands")
    df_category = spark.table(f"ecommerce.silver.slv_category")

    # Step 1: Join Brands and Categories
    df_brands_categories = (
        df_brands.alias("b")
        .join(
            df_category.alias("c"),
            F.col("b.category_code") == F.col("c.category_code"),
            "inner"
        )
        .select(
            F.col("b.brand_name"),
            F.col("b.brand_code"),
            F.col("c.category_name"),
            F.col("c.category_code")
        )
    )

    # Step 2: Denormalize with Products and format Gold Attributes
    df_gold_products = (
        df_products.alias("p")
        .join(
            df_brands_categories.alias("bc"),
            F.col("p.brand_code") == F.col("bc.brand_code"),
            "left"
        )
        .select(
            F.col("p.product_id"),
            F.col("p.sku"),
            F.col("p.category_code"),
            F.coalesce(F.col("bc.category_name"), F.lit("Not Available")).alias("category_name"),
            F.col("p.brand_code"),
            F.coalesce(F.col("bc.brand_name"), F.lit("Not Available")).alias("brand_name"),
            F.col("p.color"),
            F.col("p.size"),
            F.col("p.material"),
            F.col("p.weight_grams"),
            F.col("p.length_cm"),
            F.col("p.width_cm"),
            F.col("p.height_cm"),
            F.col("p.rating_count"),
            F.current_timestamp().alias("_gold_processed_at")
        )
    )

    # Write to Gold Layer
    (df_gold_products.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(f"ecommerce.gold.gld_dim_products")
    )
    print("Successfully written: gld_dim_products")

### BUILD GOLD CUSTOMER DIMENSION (gld_dim_customers)

In [0]:
def build_dim_customers():
    print("Building Gold Customer Dimension...")
    
    # Static Mapping for Country-State to Region
    country_state_map = {
        ("India", "MH"): "West", ("India", "GJ"): "West", ("India", "RJ"): "West",
        ("India", "KA"): "South", ("India", "TN"): "South", ("India", "TS"): "South", 
        ("India", "AP"): "South", ("India", "KL"): "South", ("India", "UP"): "North", 
        ("India", "WB"): "North", ("India", "DL"): "North",
        ("Australia", "VIC"): "SouthEast", ("Australia", "WA"): "West", 
        ("Australia", "NSW"): "East", ("Australia", "QLD"): "NorthEast",
        ("United Kingdom", "ENG"): "England", ("United Kingdom", "WLS"): "Wales", 
        ("United Kingdom", "NIR"): "Northern Ireland", ("United Kingdom", "SCT"): "Scotland",
        ("United States", "MA"): "NorthEast", ("United States", "FL"): "South", 
        ("United States", "NJ"): "NorthEast", ("United States", "CA"): "West", 
        ("United States", "NY"): "NorthEast", ("United States", "TX"): "South",
        ("United Arab Emirates", "AUH"): "Abu Dhabi", ("United Arab Emirates", "DU"): "Dubai", 
        ("United Arab Emirates", "SHJ"): "Sharjah",
        ("Singapore", "SG"): "Singapore",
        ("Canada", "BC"): "West", ("Canada", "AB"): "West", ("Canada", "ON"): "East", 
        ("Canada", "QC"): "East", ("Canada", "NS"): "East", ("Canada", "IL"): "Other"
    }

    mapping_rows = [
        F.struct(F.lit(country).alias("country"), F.lit(state).alias("state"), F.lit(region).alias("region"))
        for (country, state), region in country_state_map.items()
    ]
    df_region_mapping = spark.createDataFrame(
        [row.asDict() for row in [Row(country=c, state=s, region=r) for (c, s), r in country_state_map.items()]]
    )

    df_silver_cust = spark.table(f"ecommerce.silver.slv_customers")

    df_gold_cust = (
        df_silver_cust.alias("c")
        .join(df_region_mapping.alias("rm"), on=["country", "state"], how="left")
        .select(
            F.col("c.customer_id"),
            F.col("c.phone"),
            F.col("c.country_code"),
            F.col("c.country"),
            F.col("c.state"),
            F.coalesce(F.col("rm.region"), F.lit("Other")).alias("region"),
            F.current_timestamp().alias("_gold_processed_at")
        )
    )

    (df_gold_cust.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(f"ecommerce.gold.gld_dim_customers")
    )
    print("Successfully written: gld_dim_customers")

### BUILD GOLD DATE DIMENSION (gld_dim_date)

In [0]:
def build_dim_date():
    print("Building Gold Date Dimension...")
    
    df_silver_calendar = spark.table(f"ecommerce.silver.slv_date")

    df_gold_date = (
        df_silver_calendar
        .withColumn("date_id", F.date_format(F.col("date"), "yyyyMMdd").cast(IntegerType()))
        .withColumn("month_name", F.date_format(F.col("date"), "MMMM"))
        .withColumn("is_weekend", F.when(F.col("day_name").isin("Saturday", "Sunday"), 1).otherwise(0))
        .withColumn("_gold_processed_at", F.current_timestamp())
        .select(
            "date_id",
            "date",
            "year",
            "month_name",
            "day_name",
            "is_weekend",
            "quarter",
            "week",
            "_gold_processed_at"
        )
    )

    (df_gold_date.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(f"ecommerce.gold.gld_dim_date")
    )
    print("Successfully written: gld_dim_date")

### RUN ALL PIPELINE TASKS

In [0]:
if __name__ == "__main__":
    build_dim_products()
    build_dim_customers()
    build_dim_date()

Building Gold Product Dimension...


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7941309168581868>, line 2
      1 if __name__ == "__main__":
----> 2     build_dim_products()
      3     build_dim_customers()
      4     build_dim_date()

File <command-7941309168581862>, line 56, in build_dim_products()
     25 df_gold_products = (
     26     df_products.alias("p")
     27     .join(
   (...)     48     )
     49 )
     51 # Write to Gold Layer
     52 (df_gold_products.write
     53     .format("delta")
     54     .mode("overwrite")
     55     .option("mergeSchema", "true")
---> 56     .saveAsTable(f"ecommerce.gold.gld_dim_products")
     57 )
     58 print("Successfully written: gld_dim_products")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:828, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    826 self._write.table_nam